# Beat Tracker and Tempo Estimation
- Asa Picton, Connor Richardson, Dylan Baecker
- CSC475

### 1. Import libraries

In [ ]:

from pathlib import Path

import numpy as np

import librosa as lb

# import pywt as pwt
# from pywt import wavedec

# import optuna




### 2. Load audio and tempo data

In [29]:
audio_path = Path("giantsteps-tempo-dataset/audio")
label_path = Path("giantsteps-tempo-dataset/annotations/tempo")

# Load tempo labels
tempo_labels = {}
for file in label_path.iterdir():
    tempo_labels[file.stem] = float(file.read_text().strip())

# Build dataset
dataset = []
for file in audio_path.iterdir():
    track_id = file.stem
    if track_id in tempo_labels:
        dataset.append((file, tempo_labels[track_id]))

print(f"Loaded {len(dataset)} audio tracks with tempo labels")


Loaded 664 audio tracks with tempo labels


In [30]:
print(len(dataset))
print(dataset[0])
print(dataset[0][0].exists())

664
(PosixPath('giantsteps-tempo-dataset/audio/3226172.LOFI.mp3'), 69.0)
True


### 3. Feature Extraction

In [ ]:
def root_mean_square(y_audio, sr):
    rms_vals = lb.feature.rms(y=y_audio)[0]
    rms_features = np.array([
        np.mean(rms_vals),
        np.std(rms_vals),
        np.max(rms_vals),
        np.min(rms_vals)
    ])
    return rms_features

In [ ]:
def onset_envelope(y_audio, sr):
    onset_env = lb.onset.onset_strength(y=y_audio, sr=sr)
    onset_features = np.array([
        np.mean(onset_env),
        np.std(onset_env),
        np.max(onset_env),
            np.min(onset_env)
    ])
    return onset_features

In [ ]:
X = []
y = []

for path, bpm in dataset:

    y_audio, sr = lb.load(path)


    rms_feature_array = root_mean_square(y_audio, sr)
    onset_feature_array = onset_envelope(y_audio, sr)

    features = np.concatenate(rms_feature_array, onset_feature_array)

    X.append(features)
    y.append(bpm)